In [1]:
import torch
import torch.nn as nn
import random

device = torch.device('cuda' if torch.cuda.is_available() else "cpu")

In [2]:
#############
## Encoder ##
#############
'''
This corresponds to the "Reader" from the video. 
Its only job is to take the input sentences and output the hidden and cell states (the Context Vector).
'''

class Encode(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        
        # 1. Word Embeddings (Turning words into vectors)
        self.embedding = nn.Embedding(input_dim, emb_dim)
        
        # 2. The LSTM (The actual processing unit)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, src):
        # src shape: [sequence_len, batch_size]
        
        embedded = self.embedding(src)
        embedded = self.dropout(embedded)
        # embedded shape: [sequence_len, batch_size, emb_dim]
        
        outputs, (hidden, cell) = self.rnn(embedded)
        
        # We discard 'outputs' because the Encoder only cares about the final summary
        # 'hidden' and 'cell' here are the Context Vector (the inputs for decoder)
        return hidden, cell

In [3]:
#############
## Decoder ##
#############
'''
This corresponds to the "Writer". 
Note that it takes a single token at a time (e.g., just "Au") alongside the hidden and cell states from the previous step.
'''

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super().__init__()
        
        self.output_dim = output_dim
        
        # 1. Embedding
        self.embedding = nn.Embedding(output_dim, emb_dim)
        
        # 2. LSTM
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)
        
        # 3. Linear Layer (To predict the next word probability)
        self.fc_out = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, inputs, hidden, cell):
        # inputs shape: [batch_size] (We process one word at a time)
        
        # Add a dimension for sequence length (which is 1 here)
        inputs = inputs.unsqueeze(0)
        
        embedded = self.embedding(inputs)
        embedded = self.dropout(embedded)
        
        # Pass the inputs word + the Context Vector (hidden, cell)
        output, (hidden, cell) = self.rnn(embedded, (hidden, cell))
        
        prediction = self.fc_out(output.squeeze(0))
        
        return prediction, hidden, cell
        

In [ ]:
#####################
## Seq2Seq wrapper ##
#####################
'''
This class ties everything together. 
It handles the loop where we feed the Decoder's output back into itself as the input for the next step.
'''

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device
        
    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: Input sentence (e.g., "So long")
        # trg: Target sentence (e.g., "Au revoir")
        
        batch_size = src.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim
        
        # Tensor to store decoder outputs
        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        
        # 1. ENCODER STEP
        # Pass the source into the encoder to get the context vector
        hidden, cell = self.encoder(src)
        
        # 2. DECODER INITIALIZATION
        # The first input to the decoder is the <SOS> token (SOS = start of sequence)
        inputs = trg[0, :]
        
        # 3. DECODER LOOP
        for t in range(1, trg_len):
            
            # Pass input and context vector to decoder
            output, hidden, cell = self.decoder(inputs, hidden, cell)
            
            # Store prediction
            outputs[t] = output
            
            # TEACHER FORCING
            # Decide if we use the actual next word from data (teacher forcing)
            # or the model's predicted word.abs
            teacher_force = random.random() < teacher_forcing_ratio
            
            # Get the highest predicted token
            top1 = output.argmax(1)
            
            # If teacher forcing, next input is target token;
            # else it's predicted token
            inputs = trg[t] if teacher_force else top1
            
        return outputs